# Systemy AI – Laboratorium 1
## Python, NumPy i pierwsze metryki klasyfikacji

**Cel:** przypomnienie Pythona w kontekście pracy z wynikami modeli, samodzielna implementacja metryk klasyfikacji binarnej i pierwszy mały eksperyment z hipotezą.

**Zasady**
- Zapisz kopię notatnika na swoim Dysku: *Plik → Zapisz kopię na Dysku*, nazwa `Lab1_AI_Nazwisko.ipynb`.
- Rozwiązania wpisuj w komórkach `# Twoje rozwiązanie`, odpowiedzi opisowe w komórkach *Twoja odpowiedź*.
- Komórki oznaczone `# TEST` sprawdzają rozwiązanie. Nie zmieniaj ich; jeśli zgłaszają błąd, popraw kod powyżej.
- Przed oddaniem: *Środowisko wykonawcze → Uruchom ponownie i uruchom wszystko*. Notatnik musi wykonać się od góry do dołu bez błędów.



**Materiały pomocnicze**
- Python: https://www.w3schools.com/python/ oraz https://www.kaggle.com/learn/python
- NumPy: https://numpy.org/doc/stable/user/quickstart.html
- Metryki w scikit-learn: https://scikit-learn.org/stable/modules/model_evaluation.html

---
## Część A. Rozgrzewka

### Zadanie A1. Pierwsze komórki
1. Wypisz instrukcją `print` tekst `Hello, Artificial Intelligence!`.

In [ ]:
# Twoje rozwiązanie
print("Hello, Artificial Intelligence!")

*Twoja odpowiedź:*

### Zadanie A2. Zmienne i typy
Zadeklaruj zmienne `model`, `version`, `accuracy`, `deployed` o wartościach `"NeuralNet"`, `2`, `0.94`, `True`.

1. Wypisz typ każdej zmiennej (`type()`).
2. Zbuduj i wypisz zdanie `Model NeuralNet (v2) osiąga dokładność 0.94.` na dwa sposoby: przez konkatenację (`+`, zmienna `s1`) oraz jako f-string (zmienna `s2`).


In [ ]:
# Twoje rozwiązanie
model = "NeuralNet"
version = 2
accuracy = 0.94
deployed = True
print(type(model))
print(type(version))
print(type(accuracy))
print(type(deployed))
s1 = "Model " + model + " (v" + str(version) + ") osiąga dokładność " + str(accuracy) + "."
s2 = f"Model {model} (v{version}) osiąga dokładność {accuracy}."
print(s1)
print(s2)

In [ ]:
# TEST
assert s1 == s2 == "Model NeuralNet (v2) osiąga dokładność 0.94."
print("A2 OK")

### Zadanie A3. Średnia i odchylenie standardowe
Dla listy `acc = [0.78, 0.86, 0.92, 0.81]` oblicz **bez bibliotek**:
- średnią `mean_acc`,
- odchylenie standardowe populacji `std_acc` (z dzieleniem przez $n$):

$$\sigma = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(x_i-\bar{x})^2}$$

W komórce tekstowej zinterpretuj wynik: co mówi odchylenie o tych czterech modelach? Jak zmieni się wartość, jeśli podzielisz przez $n-1$, i kiedy stosuje się ten wariant?

In [ ]:
acc = [0.78, 0.86, 0.92, 0.81]

mean_acc = sum(acc) / len(acc)

variance = sum((x - mean_acc) ** 2 for x in acc) / len(acc)
std_acc = variance**0.5

print(f"mean_acc: {mean_acc}")
print(f"std_acc: {std_acc:.4f}")


In [ ]:
# TEST
assert abs(mean_acc - 0.8425) < 1e-9
assert abs(std_acc - 0.05309) < 1e-4, "Sprawdź, czy dzielisz przez n, a nie n-1"
print("A3 OK")

*Twoja odpowiedź:*

1. Interpretacja odchylenia:
Odchylenie standardowe wynosi około 0.0531 (5.31 p.p.) przy średniej accuracy równej 0.8425 (84.25%). Pokazuje to umiarkowane zróżnicowanie wyników między modelami. Wyniki poszczególnych modeli odbiegają od średniej przeciętnie o około 5.3%, a sam rozrzut mieści się w przedziale od 0.78 do 0.92.

2. Zmiana wartości przy dzieleniu przez n - 1:
Jeśli podzielimy przez n - 1, wartość odchylenia standardowego wzrośnie z około 0.0531 do około 0.0613. Wynika to bezpośrednio z tego, że w mianowniku dzielimy przez mniejszą liczbę, czyli przez 3 zamiast przez 4.
s
3. Kiedy stosuje się wariant n - 1:
Wariant z dzieleniem przez n - 1 (czyli tak zwaną poprawkę Bessela lub odchylenie z próby) stosuje się wtedy, gdy nasze dane stanowią jedynie próbkę pobraną z większej populacji, a nie pełną populację. Dzielenie przez n - 1 wyrównuje niedoszacowanie wariancji i daje nieobciążony wynik dla całej populacji.

### Zadanie A4. Tekst i słownik częstości
Dla tekstu

`text = "AI systems learn from data. AI systems make predictions from data"`

1. Usuń interpunkcję, zamień litery na małe i podziel tekst na słowa.
2. Zbuduj słownik `counts` w postaci `{słowo: liczba_wystąpień}`.
3. Zapisz w liście `most_common` **wszystkie** słowa o największej liczbie wystąpień (uwaga na remisy) i wypisz ją.

In [ ]:
# Twoje rozwiązanie
import string
from collections import Counter
text = "AI systems learn from data. AI systems make predictions from data"
text_cleared = ''.join(ch for ch in text.lower() if ch not in string.punctuation); 
print(text_cleared)

splitted_text = text_cleared.split()
print(splitted_text)

counts = dict(Counter(splitted_text))
print(counts)

def f_most_common(counts):
    max_val = max(counts.values())

    most_common = [word for word, count in counts.items() if count == max_val]
    print(most_common)

    return most_common

most_common = f_most_common(counts)


In [ ]:
# TEST
assert counts["data"] == 2 and counts["learn"] == 1
assert sorted(most_common) == ["ai", "data", "from", "systems"]
print("A4 OK")

### Zadanie A5. Funkcja z walidacją
Napisz funkcję `performance_label(score)`, która zamienia wynik modelu `score` ∈ [0, 1] na etykietę:

| Warunek | Etykieta |
|---|---|
| `score < 0.75` | `"weak"` |
| `0.75 ≤ score ≤ 0.90` | `"average"` |
| `score > 0.90` | `"strong"` |

Wymagania:
- progi 0.75 i 0.90 należą do `"average"`; funkcja nie zaokrągla wejścia,
- dla wartości nieliczbowej (np. `"0.8"`, `None`) funkcja zgłasza `TypeError`,
- dla liczby spoza [0, 1] funkcja zgłasza `ValueError`.

Następnie dla listy `scores = [0.72, 0.81, 0.93, 0.88]` wypisz każdy wynik z etykietą i policz średnią `mean_ok` tylko z wyników, które **nie** są `"weak"`.

In [ ]:
# Twoje rozwiązanie
import statistics

def performance_label(score):
    if type(score) not in (int, float):
        raise TypeError("Oczekiwano wartości liczbowej")
    
    if score < 0.0 or score > 1.0:
        raise ValueError("Wartość poza zakresem [0, 1]")
    
    if score < 0.75:
        return "weak"
    elif score <= 0.90:
        return "average"
    else:
        return "strong"
    

scores = [0.72, 0.81, 0.93, 0.88]
ok_scores = []

for s in scores:
    label = performance_label(s)
    print(f"Wynik: {s} -> {label}")
    if label != "weak":
        ok_scores.append(s)

mean_ok = statistics.mean(ok_scores)


In [ ]:
# TEST
assert [performance_label(s) for s in (0.74, 0.75, 0.90, 0.95)] == ["weak", "average", "average", "strong"]
assert performance_label(0) == "weak" and performance_label(1) == "strong"

for bad, exc in [(1.2, ValueError), (-0.1, ValueError), ("0.8", TypeError), (None, TypeError)]:
    try:
        performance_label(bad)
    except exc:
        pass
    else:
        raise AssertionError(f"Dla {bad!r} oczekiwano wyjątku {exc.__name__}")

assert abs(mean_ok - 0.87333) < 1e-4
print("A5 OK")

---
## Część B. Struktury danych i wyniki modeli

### Zadanie B1. Słownik wyników i ranking
1. Utwórz słownik `results`: `"SVM"` → 0.81, `"DecisionTree"` → 0.74, `"NeuralNet"` → 0.92.
2. Dodaj do niego `"Transformer"` → 0.95.
3. Oblicz średnią dokładność `mean_results` i utwórz listę `above_mean` z nazwami modeli o dokładności powyżej średniej.
4. Napisz funkcję `print_ranking(results)`, która drukuje ranking malejąco w formacie

```
1. Transformer – 0.95
2. NeuralNet – 0.92
...
Najlepszy model: Transformer – rekomendowany do wdrożenia
```

i **zwraca** nazwę najlepszego modelu. Funkcja nie może modyfikować przekazanego słownika.

In [ ]:
# Twoje rozwiązanie
results = {
    "SVM": 0.81,
    "DecisionTree": 0.74,
    "NeuralNet": 0.92,
}
results.update({"Transformer": 0.95})

mean_results = sum(results.values()) / len(results)
print(mean_results)

above_mean = {}
for [label,val] in results.items():
    if val > mean_results:
        above_mean.update({label: val})

print(above_mean)
def print_ranking(results):
    des = {k: v for k, v in sorted(results.items(), key=lambda item: item[1], reverse=True)}
    print(des)
    return next(iter(des))

print_ranking(results)

In [ ]:
# TEST
assert abs(mean_results - 0.855) < 1e-9
assert sorted(above_mean) == ["NeuralNet", "Transformer"]
assert print_ranking({"A": 0.5, "B": 0.7}) == "B"
assert list(results) == ["SVM", "DecisionTree", "NeuralNet", "Transformer"], "Funkcja zmieniła słownik"
print("B1 OK")

### Zadanie B2. Filtrowanie błędnych wyników
Dane z eksperymentów bywają zanieczyszczone. Dla listy

`raw = [0.83, 0.92, 1.05, -0.12, 0.78, None]`

napisz funkcję `clean_scores(raw)`, która zwraca parę `(valid, n_invalid)`:
- `valid` – lista poprawnych wyników (liczby z zakresu [0, 1]),
- `n_invalid` – liczba odrzuconych wartości.

Wypisz komunikat dokładnie w formacie:

`Poprawne: 3 | Błędne: 3 | Średnia (poprawnych): 0.84`

*Dla dociekliwych:* co Twoja funkcja zrobi z wartościami `"0.9"` i `True`? Czy tak powinno być?

In [ ]:
# Twoje rozwiązanie

raw = [0.83, 0.92, 1.05, -0.12, 0.78, None]

def clean_scores(raw):
    valid = []
    n_invalid = 0;
    for x in raw:
        if isinstance(x, (int, float)) and not isinstance(x,bool) and 0 <= x <= 1:
            valid.append(x)
        else:
            n_invalid += 1 
    return valid, n_invalid

raw = [0.83, 0.92, 1.05, -0.12, 0.78, None]

valid, n_invalid = clean_scores(raw)

srednia = sum(valid) / len(valid) if valid else 0

print(
    f"Poprawne: {len(valid)} | Błędne: {n_invalid} | Średnia (poprawnych): {srednia:.2f}"
)

In [ ]:
# TEST
assert clean_scores([0.83, 0.92, 1.05, -0.12, 0.78, None]) == ([0.83, 0.92, 0.78], 3)
assert clean_scores([0, 1, "0.9"]) == ([0, 1], 1)
print("B2 OK")

### Zadanie B3. Struktura zagnieżdżona
Dla struktury `model_info` (komórka niżej) napisz funkcję `summarize_model(info)`, która:
1. wypisuje nazwę modelu,
2. oblicza i wypisuje średnią ze **wszystkich** metryk w `info["metrics"]` (nie zakładaj, że są dokładnie trzy),
3. wypisuje `Model gotowy do wdrożenia`, jeśli średnia ≥ 0.9, w przeciwnym razie `Model wymaga poprawy`,
4. zapisuje średnią pod kluczem `info["mean_score"]` i zwraca cały słownik.

Wywołaj funkcję: `model_info = summarize_model(model_info)`.

In [ ]:
model_info = {
    "name": "NeuralNet",
    "version": "1.2",
    "metrics": {"accuracy": 0.91, "precision": 0.88, "recall": 0.85},
    "status": "trained",
}

In [ ]:
# Twoje rozwiązanie
def summarize_model(info):
    mean = 0
    for metric in info["metrics"].values():
        mean += metric
    mean /= len(info["metrics"])
    info["mean_score"] = mean

    print(f"{info['name']} {info['mean_score']}")

    if mean >= 0.9:
        print("Model gotowy do wdrożenia")
    else:
        print("Model wymaga poprawy")

    return info
    
    
    
    
model_info = summarize_model(model_info)

In [ ]:
# TEST
assert abs(model_info["mean_score"] - 0.88) < 1e-9
assert abs(summarize_model({"name": "X", "metrics": {"accuracy": 0.95, "f1": 0.93}})["mean_score"] - 0.94) < 1e-9
print("B3 OK")

---
## Część C. Metryki klasyfikacji binarnej

Przypomnienie (klasa pozytywna = 1):

| | predykcja 1 | predykcja 0 |
|---|---|---|
| **prawda 1** | TP | FN |
| **prawda 0** | FP | TN |

$$\text{accuracy}=\frac{TP+TN}{TP+TN+FP+FN}\qquad \text{precision}=\frac{TP}{TP+FP}\qquad \text{recall}=\frac{TP}{TP+FN}\qquad F_1=\frac{2\cdot\text{precision}\cdot\text{recall}}{\text{precision}+\text{recall}}$$

In [ ]:
y_true = [1, 0, 1, 1, 0, 0, 1, 0, 1, 0]
y_pred = [1, 0, 1, 0, 0, 1, 1, 0, 0, 0]

### Zadanie C1. Ręcznie, w czystym Pythonie
Używając pętli (bez bibliotek) policz i zapisz w zmiennych `TP`, `FP`, `FN`, `TN`, a następnie `precision`, `recall` i `f1`. Wypisz raport:

`TP=.. FP=.. FN=.. TN=.. / precision=.. recall=.. F1=..`

In [ ]:
# Twoje rozwiązanie
TP = 0
FP = 0
FN = 0
TN = 0

for true, pred in zip(y_true, y_pred):
    if true == 1 and pred == 1:
        TP += 1
    elif true == 0 and pred == 1:
        FP += 1
    elif true == 1 and pred == 0:
        FN += 1
    elif true == 0 and pred == 0:
        TN += 1
        
accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0
precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(
    f"TP={TP}. FP={FP}. FN={FN}. TN={TN}. / precision={precision:.2f}. recall={recall:.2f}. F1={f1:.2f}."
)

In [ ]:
# TEST
assert (TP, FP, FN, TN) == (3, 1, 2, 4)
assert abs(precision - 0.75) < 1e-9 and abs(recall - 0.6) < 1e-9 and abs(f1 - 2 / 3) < 1e-9
print("C1 OK")

### Zadanie C2. To samo w NumPy
Napisz funkcję `binary_metrics(y_true, y_pred)`, która **bez pętli** (maski logiczne i `sum`) zwraca słownik z kluczami

`"TP", "FP", "FN", "TN", "accuracy", "precision", "recall", "f1"`.

Jeśli mianownik którejś metryki jest równy 0 (np. model ani razu nie przewidział klasy 1), przyjmij dla niej wartość `0.0`. Funkcja ma działać zarówno dla list, jak i tablic NumPy.

In [ ]:
import numpy as np

def binary_metrics(y_true, y_pred):
    y_t = np.array(y_true)
    y_p = np.array(y_pred)

    TP = np.sum((y_t == 1) & (y_p == 1))
    FP = np.sum((y_t == 0) & (y_p == 1))
    FN = np.sum((y_t == 1) & (y_p == 0))
    TN = np.sum((y_t == 0) & (y_p == 0))

    accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0.0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )
    
    return {
        "TP": int(TP),
        "FP": int(FP),
        "FN": int(FN),
        "TN": int(TN),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }


In [ ]:
# TEST
m = binary_metrics(y_true, y_pred)
assert (m["TP"], m["FP"], m["FN"], m["TN"]) == (3, 1, 2, 4)
assert abs(m["accuracy"] - 0.7) < 1e-9 and abs(m["f1"] - 2 / 3) < 1e-9

m0 = binary_metrics([1, 0, 0, 0], [0, 0, 0, 0])   # model nigdy nie przewiduje 1
assert m0["precision"] == 0.0 and m0["recall"] == 0.0 and m0["f1"] == 0.0
print("C2 OK")

### Zadanie C3. Porównanie ze scikit-learn
Policz te same wielkości funkcjami `confusion_matrix`, `accuracy_score`, `precision_score`, `recall_score`, `f1_score` z modułu `sklearn.metrics` i sprawdź (`np.isclose`), że zgadzają się z wynikiem `binary_metrics`.

W komentarzu zapisz, w jakim układzie `confusion_matrix` zwraca TP, FP, FN, TN. Sprawdź to w dokumentacji, nie zgaduj.

In [ ]:
# Twoje rozwiązanie
import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# Układ confusion_matrix w scikit-learn:
# Według dokumentacji scikit-learn, macierz pomyłek ma postać:
# [[TN, FP],
#  [FN, TP]]
# Gdzie wiersze reprezentują wartości rzeczywiste (prawda 0, prawda 1),
# a kolumny wartości przewidywane (predykcja 0, predykcja 1).
# Spłaszczenie macierzy metodą .ravel() zwraca elementy w kolejności: TN, FP, FN, TP.

cm = confusion_matrix(y_true, y_pred)
tn_sk, fp_sk, fn_sk, tp_sk = cm.ravel()

acc_sk = accuracy_score(y_true, y_pred)
prec_sk = precision_score(y_true, y_pred)
rec_sk = recall_score(y_true, y_pred)
f1_sk = f1_score(y_true, y_pred)

bm = binary_metrics(y_true, y_pred)

assert np.isclose(bm["TP"], tp_sk)
assert np.isclose(bm["FP"], fp_sk)
assert np.isclose(bm["FN"], fn_sk)
assert np.isclose(bm["TN"], tn_sk)
assert np.isclose(bm["accuracy"], acc_sk)
assert np.isclose(bm["precision"], prec_sk)
assert np.isclose(bm["recall"], rec_sk)
assert np.isclose(bm["f1"], f1_sk)

print("C3 OK")

---
## Część D. Mini-eksperyment: kiedy accuracy wprowadza w błąd

Rozważ dwa „modele” wykrywające rzadkie zjawisko (np. chorobę, awarię, oszustwo):

- **zawsze-zero** – niezależnie od wejścia odpowiada 0,
- **zaszumiony** – zna prawdziwą odpowiedź, ale w 10% przypadków (losowo) podaje przeciwną.

Parametr $p$ to udział klasy pozytywnej w danych.

### Zadanie D1. Hipoteza
**Zanim uruchomisz jakikolwiek kod**, zapisz przewidywania dla $p = 0.05$:
- jaką accuracy osiągnie każdy z modeli?
- który model będzie miał wyższą accuracy?
- jaką precision będzie miał model zaszumiony: bliżej 0.9, 0.5 czy 0.3?

*Twoja odpowiedź:*
Dla p = 0.05:
1. Model zawsze-zero osiągnie accuracy równe 0.95 (95%), a model zaszumiony accuracy równe 0.90 (90%).
2. Wyższe accuracy będzie miał model zawsze-zero (95% vs 90%).
3. Precision dla modelu zaszumionego będzie wynosić bliżej 0.3, a dokładnie około 0.321.

Funkcja poniżej generuje dane. Jest gotowa, przeczytaj ją i upewnij się, że rozumiesz każdą linię.

In [ ]:
import numpy as np

def make_data(p, n=10_000, error_rate=0.10, seed=42):
    """Zwraca (y_true, y_zero, y_noisy) dla udziału klasy pozytywnej p."""
    rng = np.random.default_rng(seed)
    y_true = (rng.random(n) < p).astype(int)        # 1 z prawdopodobieństwem p
    y_zero = np.zeros(n, dtype=int)                 # model zawsze-zero
    flip = rng.random(n) < error_rate               # które odpowiedzi odwrócić
    y_noisy = np.where(flip, 1 - y_true, y_true)    # model zaszumiony
    return y_true, y_zero, y_noisy

### Zadanie D2. Eksperyment
1. Dla $p = 0.05$ policz funkcją `binary_metrics` metryki obu modeli i porównaj z przewidywaniami z D1.
2. Powtórz obliczenia dla `ps = [0.5, 0.3, 0.1, 0.05, 0.01]`. Wyniki zbierz w tabeli `pandas.DataFrame` o nazwie `df` z kolumnami `p`, `model`, `accuracy`, `precision`, `recall`, `f1` (po jednym wierszu na parę: wartość $p$, model).
3. Narysuj wykres (matplotlib): oś X to $p$ w skali logarytmicznej, na osi Y accuracy obu modeli oraz precision modelu zaszumionego. Podpisz osie i dodaj legendę.

In [ ]:
import pandas as pd
import numpy as np


def binary_metrics(y_true, y_pred):
    y_t = np.array(y_true)
    y_p = np.array(y_pred)

    TP = np.sum((y_t == 1) & (y_p == 1))
    FP = np.sum((y_t == 0) & (y_p == 1))
    FN = np.sum((y_t == 1) & (y_p == 0))
    TN = np.sum((y_t == 0) & (y_p == 0))

    accuracy = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0.0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return {
        "TP": int(TP),
        "FP": int(FP),
        "FN": int(FN),
        "TN": int(TN),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }


ps = [0.5, 0.3, 0.1, 0.05, 0.01]
rows = []

for p in ps:
    y_true, y_zero, y_noisy = make_data(p)

    m_zero = binary_metrics(y_true, y_zero)
    rows.append(
        {
            "p": p,
            "model": "zawsze-zero",
            "accuracy": m_zero["accuracy"],
            "precision": m_zero["precision"],
            "recall": m_zero["recall"],
            "f1": m_zero["f1"],
        }
    )

    m_noisy = binary_metrics(y_true, y_noisy)
    rows.append(
        {
            "p": p,
            "model": "zaszumiony",
            "accuracy": m_noisy["accuracy"],
            "precision": m_noisy["precision"],
            "recall": m_noisy["recall"],
            "f1": m_noisy["f1"],
        }
    )

df = pd.DataFrame(rows)

print("Porównanie wyników dla p = 0.05:")
print(df[df["p"] == 0.05][["model", "accuracy", "precision", "recall", "f1"]])


In [ ]:
import matplotlib.pyplot as plt

df_zero = df[df["model"] == "zawsze-zero"]
df_noisy = df[df["model"] == "zaszumiony"]

plt.figure(figsize=(8, 5))

plt.plot(df_zero["p"], df_zero["accuracy"], marker="o", label="Accuracy: zawsze-zero")
plt.plot(df_noisy["p"], df_noisy["accuracy"], marker="s", label="Accuracy: zaszumiony")

plt.plot(
    df_noisy["p"], df_noisy["precision"], marker="^", label="Precision: zaszumiony"
)

plt.xscale("log")
plt.xlabel("Parametr p (skala logarytmiczna)")
plt.ylabel("Wartość metryki")
plt.title("Eksperyment: Wpływ niezrównoważenia klas (p) na metryki")
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend()

plt.show()

In [ ]:
# TEST
assert set(df.columns) >= {"p", "model", "accuracy", "precision", "recall", "f1"}
assert len(df) == 10
z = df[(df["model"] == "zawsze-zero") & (df["p"] == 0.01)].iloc[0]
assert z["accuracy"] > 0.98 and z["recall"] == 0.0
print("D2 OK")

### Zadanie D3. Wnioski
Odpowiedz krótko (2–3 zdania na pytanie), odwołując się do liczb z tabeli:

1. Czy Twoje przewidywania z D1 się potwierdziły? Co Cię zaskoczyło?
2. Dla $p = 0.01$: który model ma wyższą accuracy, a który jest bardziej użyteczny? Dlaczego?
3. Model zaszumiony myli się zawsze w 10% przypadków. Dlaczego mimo to jego precision spada, gdy $p$ maleje?
4. Projektujesz system wykrywający rzadką chorobę. Którą metrykę zaraportujesz jako główną? Jakie konsekwencje dla pacjentów mają błędy FP, a jakie FN?

*Twoja odpowiedź:*
1. Tak, przewidywania z D1 w pełni się potwierdziły. Dla p = 0.05 model zawsze-zero osiągnął wyższe accuracy (0.9492) niż model zaszumiony (0.9036), a precyzja zaszumionego wyszła 0.3367, czyli blisko 0.3. Zaskakuje to, że całkowicie bezużyteczny model, który nic nie robi, ma wyższą dokładność niż model, który dobrze wykrywa większość pozytywnych przypadków.

2. Dla p = 0.01 wyższe accuracy ma model zawsze-zero (około 0.99 vs 0.90). Zdecydowanie bardziej użyteczny jest jednak model zaszumiony, bo nadal wykrywa około 90% szukanych zdarzeń (recall około 0.90), podczas gdy model zawsze-zero ma recall równy 0.0000 i nie wykrywa zupełnie nic.   

3. Gdy p maleje, w zbiorze jest coraz mniej prawdziwych jedynek (TP). Jednocześnie przy ogromnej liczbie zer stały 10% błąd powoduje powstanie bardzo dużej liczby fałszywych alarmów (FP). W efekcie we wzorze Precision = TP / (TP + FP) wartość FP zaczyna dominować w mianowniku, przez co precyzja drastycznie spada z około 0.90 dla p = 0.5 do zaledwie około 0.09 dla p = 0.01.

4. Jako główną metrykę zaraportuję Recall lub F1-score, bo najważniejsze jest wykrycie jak największej liczby chorych pacjentów. Błąd FN, czyli niewykrycie choroby, jest bardzo groźny, bo pacjent nie dostanie leczenia i jego stan zdrowia się pogorszy. Błąd FP, czyli fałszywy alarm, powoduje głównie niepotrzebny stres i wymaga zrobienia dodatkowych badań.

---
## Część E. Zadania dodatkowe (dla chętnych / do domu)

**E1. Pliki.** Zapisz listę wyników (po jednym w linii) do pliku `scores.txt`. Wczytaj plik, oblicz średnią i zapisz do `report.txt` raport w formacie `Wczytano N wyników, średnia: X` (X z dokładnością do 3 miejsc po przecinku).

**E2. Dane od użytkownika.** Pobierz funkcją `input()` wynik modelu `s` i wypisz:
- `Podano niepoprawną wartość!`, gdy tekstu nie da się zamienić na liczbę (`try`/`except`),
- `Wynik poza zakresem 0–1`, gdy `s < 0` lub `s > 1`,
- `Model wymaga poprawy`, gdy `s < 0.7`,
- `Wynik akceptowalny`, gdy `0.7 ≤ s < 0.85`,
- `Wynik doskonały` w pozostałych przypadkach.

**E3. Duplikaty.** Usuń duplikaty z listy `["SVM", "SVM", "NeuralNet", "NeuralNet", "Transformer"]` **z zachowaniem kolejności** pierwszych wystąpień. Dlaczego samo `list(set(...))` tego nie gwarantuje?

**E4. Uogólnienie.** Rozszerz `make_data` o parametr pozwalający ustawić osobno odsetek błędów dla przykładów pozytywnych i negatywnych. Jak zmienia się wykres z D2, gdy model myli się w 10% pozytywnych, ale tylko w 1% negatywnych?

In [ ]:
# Twoje rozwiązanie